In [ ]:
# =========================================================
# Task05 改2: 落下→アーム先端でキャッチ保持 / 末尾を自然終了
# 75秒 / 15fps / 640x480 / Colab無料枠 / 出力: /content/task05_catch2.mp4
# =========================================================
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Rectangle
import os

FPS=15; DURATION=75.0; NFRAMES=int(FPS*DURATION)
rng=np.random.default_rng(42)
BASE=(0.0,0.2); L1,L2=0.8,0.6; G=9.8
DROP_H=2.8; DROP_INTERVAL=3.0
CATCH_R=0.12; HOLD_TIME=1.2
HOME=(0.0,1.2)                       # 待機時のアーム先端位置
FALL_TIME=np.sqrt(2*(DROP_H-0.6)/G)  # おおよその落下時間
SAFE_MARGIN=HOLD_TIME+FALL_TIME+0.5  # 末尾で新規ドロップしない余白

def ik_2link(target, base=BASE, L1=L1, L2=L2):
    dx,dy=target[0]-base[0],target[1]-base[1]
    d=np.hypot(dx,dy); d=min(d,L1+L2-1e-3); d=max(d,abs(L1-L2)+1e-3)
    a=np.arctan2(dy,dx)
    cos1=np.clip((L1**2+d**2-L2**2)/(2*L1*d),-1,1); th1=a-np.arccos(cos1)
    cos2=np.clip((L1**2+L2**2-d**2)/(2*L1*L2),-1,1); th2=np.pi-np.arccos(cos2)
    return th1,th2

def fk_2link(th1,th2,base=BASE,L1=L1,L2=L2):
    x1=base[0]+L1*np.cos(th1); y1=base[1]+L1*np.sin(th1)
    x2=x1+L2*np.cos(th1+th2);  y2=y1+L2*np.sin(th1+th2)
    return np.array([base,(x1,y1),(x2,y2)])

class BallPredictor:
    def __init__(self,g=G,win=6): self.g=g; self.win=win; self.hist=[]
    def reset(self): self.hist.clear()
    def update(self,t,y):
        self.hist.append((t,y))
        if len(self.hist)>self.win: self.hist.pop(0)
    def predict_y_at(self,t_future):
        if len(self.hist)<3: return None
        ts=np.array([h[0] for h in self.hist]); ys=np.array([h[1] for h in self.hist])
        A=np.c_[np.ones_like(ts),ts,-0.5*self.g*ts**2]
        coef,*_=np.linalg.lstsq(A,ys,rcond=None); y0,v0=coef[0],coef[1]
        return y0+v0*t_future-0.5*self.g*t_future**2

fig,ax=plt.subplots(figsize=(6.4,4.8),dpi=100)
ax.set_xlim(-3,3); ax.set_ylim(-0.5,3); ax.set_aspect('equal'); ax.grid(alpha=0.2)
ax.add_patch(Rectangle((-3,-0.1),6,0.1,color='0.6'))
ax.plot([BASE[0]],[BASE[1]],'s',color='k',ms=6)

arm_line,=ax.plot([],[],'-o',color='C0',lw=3,ms=5)
ball=ax.add_patch(Circle((0,0),0.09,color='C3'))
status=ax.text(-2.8,2.75,"",fontsize=9,color='C3')
count_txt=ax.text(1.6,2.75,"caught: 0",fontsize=10,color='C2',fontweight='bold')
ax.set_title("Task05: Predictive Catch & Hold (75s)",fontsize=10)

T=np.arange(NFRAMES)/FPS
state={"drop_t":0.0,"drop_x":rng.uniform(-1.5,1.5),"next_drop":0.0,
       "caught":0,"holding":False,"hold_until":0.0,"hold_pos":(0,0),
       "active":False}

pred=BallPredictor()

def set_arm(j):
    arm_line.set_data(j[:,0],j[:,1]); return j

def animate(i):
    t=T[i]

    # --- 保持中 ---
    if state["holding"]:
        th1,th2=ik_2link(state["hold_pos"]); j=set_arm(fk_2link(th1,th2))
        ball.center=state["hold_pos"]; status.set_text("HOLDING")
        if t>=state["hold_until"]:
            state["holding"]=False; state["active"]=False
        return arm_line,ball,status,count_txt

    # --- 新規ドロップ（末尾余白を考慮） ---
    if (not state["active"]) and (t+SAFE_MARGIN < DURATION) and (t>=state["next_drop"]):
        state["drop_t"]=t; state["drop_x"]=rng.uniform(-1.5,1.5)
        state["next_drop"]=t+DROP_INTERVAL; state["active"]=True; pred.reset()

    # --- 待機（ドロップなし or 末尾） ---
    if not state["active"]:
        th1,th2=ik_2link(HOME); set_arm(fk_2link(th1,th2))
        ball.center=(HOME[0],HOME[1]); status.set_text("standby")
        return arm_line,ball,status,count_txt

    # --- 落下中 ---
    tt=t-state["drop_t"]; y=DROP_H-0.5*G*tt**2
    if y<0: y=0
    else: pred.update(tt,y)

    tip_y=max(0.6,min(y,1.8))
    th1,th2=ik_2link((state["drop_x"],tip_y)); j=set_arm(fk_2link(th1,th2))

    tip=j[2]; dist=np.hypot(tip[0]-state["drop_x"],tip[1]-y)
    if dist<CATCH_R:
        state["holding"]=True; state["hold_until"]=t+HOLD_TIME
        state["hold_pos"]=(tip[0],tip[1]); state["caught"]+=1
        count_txt.set_text(f"caught: {state['caught']}")
        ball.center=state["hold_pos"]; status.set_text("CAUGHT!")
        return arm_line,ball,status,count_txt

    ball.center=(state["drop_x"],y)
    status.set_text(f"drop x={state['drop_x']:+.2f}  y={y:.2f}")
    return arm_line,ball,status,count_txt

anim=FuncAnimation(fig,animate,frames=NFRAMES,interval=1000/FPS,blit=False)
out="/content/task05_catch2.mp4"
anim.save(out,writer=FFMpegWriter(fps=FPS,bitrate=1200),dpi=100)
plt.close(fig); print("saved:",out,f"{os.path.getsize(out)/1e6:.2f} MB")
from IPython.display import Video; Video(out,embed=True,width=640)